# Robust Learning: bounded CIFAR-10 GPU timing pilot

Runs one full-data warmup epoch and one mature-operation epoch for DSS+ and DivideMix, plus DivideMix's full-data GMM scan. The entire pilot has a shared 30-minute deadline and records censored stages. It measures timing and GPU memory only; it does not evaluate accuracy, selection quality, or research hypotheses.

Before running, attach an existing Kaggle dataset containing the verified official `cifar-10-batches-py` directory. Enable GPU. No internet, credentials, Kaggle CLI, dataset upload, or dependency installation is used by this notebook. Review `research/kaggle_gpu_pilot.md` in the repository for fidelity decisions and limitations. The bundled backbone is the MIT-licensed DivideMix `PreResNet.py`; the original license is embedded alongside it. DSS source is not bundled.


In [ ]:
from pathlib import Path
destination = Path('/kaggle/working/robust_learning_research') / 'configs/strong_baseline_pilot.json'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text('{\n  "name": "strong-baseline-20261012",\n  "device": "cpu",\n  "threads": 2,\n  "pilot_seed": 601,\n  "split_seed": 1601,\n  "noise_seed": 2601,\n  "validation_per_class": 500,\n  "imbalance_factor": 10,\n  "noise_kind": "symmetric_including_original",\n  "noise_rate": 0.5,\n  "pilot_batch_size": 16,\n  "pilot_samples": 128,\n  "warmup_microepochs": 4,\n  "timed_updates_per_method": 3,\n  "max_training_seconds": 180,\n  "max_process_rss_mib": 1800,\n  "minimum_available_memory_mib": 700,\n  "methods": ["ce", "classwise_small_loss", "dss_uniform", "dss_estimated_prior", "dss_plus_estimated_prior", "dividemix"],\n  "dss": {"epochs": 150, "warmup": 30, "momentum": 0.9, "weight_decay": 0.001, "lr": 0.02, "lr_drop_epoch": 80, "marginal_decay": 0.99, "alpha": 0.1},\n  "dss_plus": {"epochs": 300, "warmup": 20, "momentum": 0.9, "weight_decay": 0.0005, "lr": 0.02, "minimum_lr": 0.0004, "marginal_decay": 0.99, "alpha": 0.1, "temperature": 2.0, "mixup_beta": 4.0, "consistency_weight": 2.0},\n  "dividemix": {"epochs": 300, "warmup": 10, "momentum": 0.9, "weight_decay": 0.0005, "lr": 0.02, "lr_drop_epoch": 150, "temperature": 0.5, "mixup_alpha": 4.0, "clean_threshold": 0.5, "lambda_u": 25.0, "lambda_u_ramp_epochs": 16, "regularizer_weight": 1.0},\n  "future_development_seeds": [701, 702],\n  "future_evaluation_seeds": [801, 802, 803, 804, 805],\n  "future_equal_candidates_per_method": 2,\n  "future_regimes": ["balanced_sym50", "longtail10_sym50", "longtail10_asym40"],\n  "automatic_full_campaign": false,\n  "approved_full_campaign_hours": 0\n}\n', encoding='utf-8')


In [ ]:
from pathlib import Path
destination = Path('/kaggle/working/robust_learning_research') / 'src/robust_learning/__init__.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text('"""Research-discovery controls; no novel method implemented."""\n', encoding='utf-8')


In [ ]:
from pathlib import Path
destination = Path('/kaggle/working/robust_learning_research') / 'src/robust_learning/strong/__init__.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text('"""Published-method equation checks and bounded real-image feasibility tools."""\n', encoding='utf-8')


In [ ]:
from pathlib import Path
destination = Path('/kaggle/working/robust_learning_research') / 'src/robust_learning/strong/cifar.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text('"""Benchmark construction separates true labels from observed-label training."""\nfrom dataclasses import dataclass\n\nimport numpy as np\nfrom PIL import Image\nfrom torchvision import transforms\n\n\n@dataclass(frozen=True)\nclass ImageTrainingView:\n    images: np.ndarray\n    observed_y: np.ndarray\n    ids: np.ndarray\n\n    def __post_init__(self):\n        for array in (self.images, self.observed_y, self.ids):\n            array.setflags(write=False)\n\n\ndef make_splits(labels, seed, validation_per_class=500, imbalance_factor=10,\n                prior_majority=100, model_val_per_class=200):\n    labels = np.asarray(labels)\n    if imbalance_factor < 1 or validation_per_class < prior_majority+model_val_per_class:\n        raise ValueError("invalid prior ratio/validation reservation")\n    rng = np.random.default_rng(np.random.SeedSequence([seed, 401]))\n    train, validation, prior = [], [], []\n    for c in range(10):\n        available = rng.permutation(np.where(labels == c)[0])\n        if len(available) <= validation_per_class:\n            raise ValueError("insufficient per-class images")\n        reserve = available[:validation_per_class]\n        n_prior = max(1, int(prior_majority*imbalance_factor**(-c/9)))\n        prior.extend(reserve[:n_prior])\n        validation.extend(reserve[n_prior:n_prior+model_val_per_class])\n        pool = available[validation_per_class:]\n        train.extend(pool[:max(1, int(len(pool)*imbalance_factor**(-c/9)))])\n    return {"train": np.array(train, dtype=np.int64), "validation": np.array(validation, dtype=np.int64),\n            "prior": np.array(prior, dtype=np.int64)}\n\n\ndef corrupt(true, seed, rate, kind):\n    if not 0 <= rate <= 1:\n        raise ValueError("noise rate must be in [0,1]")\n    true = np.asarray(true)\n    rng = np.random.default_rng(np.random.SeedSequence([seed, 991]))\n    intervention = rng.random(len(true)) < rate\n    observed = true.copy()\n    if kind == "symmetric_including_original":\n        observed[intervention] = rng.integers(0, 10, intervention.sum())\n    elif kind == "asymmetric":\n        transition = np.array([0, 1, 0, 5, 7, 3, 6, 7, 8, 1])\n        observed[intervention] = transition[true[intervention]]\n    else:\n        raise ValueError("unregistered noise family")\n    return observed, observed != true, intervention\n\n\ndef image_transforms():\n    mean, std = (.4914, .4822, .4465), (.2023, .1994, .2010)\n    weak = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),\n                               transforms.ToTensor(), transforms.Normalize(mean, std)])\n    strong = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),\n        transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10), transforms.ToTensor(), transforms.Normalize(mean, std)])\n    inference = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])\n    return weak, strong, inference\n\n\ndef batch_images(view, indices, transform):\n    import torch\n    return torch.stack([transform(Image.fromarray(view.images[i])) for i in indices])\n', encoding='utf-8')


In [ ]:
from pathlib import Path
destination = Path('/kaggle/working/robust_learning_research') / 'src/robust_learning/strong/equations.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text('"""Independent equations from DSS and DivideMix; no unlicensed source copied."""\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.mixture import GaussianMixture\n\n\ndef adjusted_probability(probability, marginal, prior):\n    if not torch.isfinite(prior).all() or (prior <= 0).any() or (marginal <= 0).any():\n        raise ValueError("strictly positive finite prior/marginal required")\n    adjusted = probability*prior/marginal\n    return adjusted/adjusted.sum(-1, keepdim=True)\n\n\ndef mann_kendall_z(history):\n    """History [epochs, examples, classes]; paper\'s no-tie-adjustment variance."""\n    n = len(history)\n    if n < 2:\n        return torch.zeros_like(history[0])\n    score = torch.zeros_like(history[0])\n    for t in range(1, n):\n        score += torch.sign(history[t]-history[:t]).sum(0)\n    variance = n*(n-1)*(2*n+5)/18\n    return (score-torch.sign(score))/variance**.5\n\n\ndef candidate_classes(history, observed, threshold):\n    exclude = mann_kendall_z(history) > threshold\n    exclude[torch.arange(len(observed)), observed] = False\n    return exclude\n\n\ndef ccs_loss(logits, observed, exclude, keep):\n    if exclude[torch.arange(len(observed)), observed].any():\n        raise ValueError("annotated class must remain in denominator")\n    modified = logits.masked_fill(exclude, -torch.inf)\n    per_example = F.cross_entropy(modified, observed, reduction="none")\n    return (per_example*keep).mean()\n\n\ndef sharpen(probability, temperature):\n    if temperature <= 0:\n        raise ValueError("positive temperature required")\n    powered = probability**(1/temperature)\n    return powered/powered.sum(-1, keepdim=True)\n\n\ndef refine_targets(observed, probability, clean_weight, temperature):\n    label = F.one_hot(observed, probability.shape[-1]).to(probability.dtype)\n    refined = clean_weight[:, None]*label+(1-clean_weight[:, None])*probability\n    return sharpen(refined, temperature).detach()\n\n\ndef semisupervised_loss(logits_x, target_x, logits_u, target_u, mse_class_reduction="mean"):\n    """Official DivideMix class-mean MSE; paper Eq.10 class-sum selectable.\n\n    The difference changes effective lambda_u by the class count. Preserve and\n    disclose the implementation convention instead of silently conflating them.\n    """\n    lx = -(target_x*F.log_softmax(logits_x, dim=1)).sum(1).mean()\n    difference = (logits_u.softmax(1)-target_u)**2\n    if mse_class_reduction not in {"mean", "sum"}:\n        raise ValueError("MSE class reduction must be mean or sum")\n    lu = difference.mean() if mse_class_reduction == "mean" else difference.sum(1).mean()\n    prediction = torch.cat([logits_x, logits_u]).softmax(1).mean(0)\n    prior = torch.ones_like(prediction)/len(prediction)\n    reg = (prior*(prior.log()-prediction.clamp_min(1e-12).log())).sum()\n    return lx, lu, reg\n\n\ndef fit_clean_probability(loss, seed):\n    loss = np.asarray(loss, dtype=np.float64)\n    if len(loss) < 4 or not np.isfinite(loss).all() or np.ptp(loss) <= 1e-12:\n        raise ValueError("nondegenerate finite observed-loss scan required")\n    scaled = ((loss-loss.min())/np.ptp(loss)).reshape(-1, 1)\n    mixture = GaussianMixture(n_components=2, max_iter=10, tol=.01, reg_covar=.0005, random_state=seed)\n    mixture.fit(scaled)\n    return mixture.predict_proba(scaled)[:, mixture.means_.argmin()]\n\n\ndef classwise_keep(loss, observed, fraction):\n    keep = torch.zeros(len(loss), dtype=torch.bool)\n    for c in observed.unique():\n        indices = torch.where(observed == c)[0]\n        selected = indices[torch.argsort(loss[indices], stable=True)[:max(1, int(fraction*len(indices)))]]\n        keep[selected] = True\n    return keep\n\n\ndef peer_policies(first, second):\n    """Exchange frozen policies, never self-select in a two-network update."""\n    return second, first\n', encoding='utf-8')


In [ ]:
from pathlib import Path
destination = Path('/kaggle/working/robust_learning_research') / 'src/robust_learning/strong/kernels.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text('"""Independent paper-equation update kernels; no training-label oracle inputs."""\nimport torch\nimport torch.nn.functional as F\n\nfrom .cifar import batch_images\nfrom .equations import adjusted_probability, ccs_loss, refine_targets, sharpen, semisupervised_loss\n\n\nclass DSSState:\n    def __init__(self, examples, classes, prior, decay=.99):\n        self.prior = prior\n        self.decay = decay\n        self.marginal = torch.ones(classes)/classes\n        self.current = torch.ones(examples, classes)/classes\n        self.history = []\n        self.score = torch.zeros(examples, classes)\n\n    def observe(self, indices, probability):\n        # Histories stay on the host; only the current minibatch occupies VRAM.\n        probability = probability.detach().cpu()\n        indices = torch.as_tensor(indices, dtype=torch.long, device="cpu")\n        self.marginal = self.decay*self.marginal+(1-self.decay)*probability.mean(0)\n        self.current[indices] = adjusted_probability(probability, self.marginal, self.prior).detach()\n\n    def finish_epoch(self):\n        if self.history:\n            self.score += torch.sign(self.current-torch.stack(self.history)).sum(0)\n        self.history.append(self.current.clone())\n\n    def policy(self, observed, threshold=1.2815515655446004):\n        keep = self.current.argmax(1) == observed\n        t = len(self.history)\n        variance = t*(t-1)*(2*t+5)/18\n        z = (self.score-self.score.sign())/variance**.5 if t >= 2 else torch.zeros_like(self.score)\n        exclude = z > threshold\n        exclude[torch.arange(len(observed)), observed] = False\n        return keep, exclude\n\n\ndef _mix(images, targets, mix, generator):\n    permutation = torch.randperm(len(images), generator=generator).to(images.device)\n    return mix*images+(1-mix)*images[permutation], mix*targets+(1-mix)*targets[permutation]\n\n\ndef mixup(images, targets, alpha, rng, generator):\n    draw = float(rng.beta(alpha, alpha))\n    return _mix(images, targets, max(draw, 1-draw), generator)\n\n\ndef dss_update(model, optimizer, state, indices, weak, observed, keep, exclude,\n               strong=None, plus_config=None, rng=None, generator=None):\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    logits = model(weak)\n    probability = logits.softmax(1).detach()\n    state.observe(indices, probability)\n    supervised = ccs_loss(logits, observed, exclude.to(logits.device), keep.to(logits.device))\n    total = supervised\n    consistency = logits.new_zeros(())\n    if strong is not None:\n        # DSS+ supplement Eq.6 uses p^(1/T); official code uses p^T and\n        # adjusted targets. This kernel follows the published raw-probability equation.\n        target = sharpen(probability, plus_config["temperature"]).detach()\n        mixed, mixed_target = mixup(strong, target, plus_config["mixup_beta"], rng, generator)\n        consistency = -(mixed_target*model(mixed).log_softmax(1)).sum(1).mean()\n        total = supervised+plus_config["consistency_weight"]*consistency\n    total.backward()\n    optimizer.step()\n    return {"loss": float(total.detach()), "supervised_loss": float(supervised.detach()),\n            "consistency_loss": float(consistency.detach()), "accepted_labels": int(keep.sum())}\n\n\ndef dividemix_update(model, peer, optimizer, view, labeled_indices, unlabeled_indices,\n                     peer_probability, weak_transform, config, rng, generator,\n                     ramp_fraction=1.):\n    model.train(); peer.eval()\n    device = next(model.parameters()).device\n    x1, x2 = [batch_images(view, labeled_indices, weak_transform).to(device) for _ in range(2)]\n    u1, u2 = [batch_images(view, unlabeled_indices, weak_transform).to(device) for _ in range(2)]\n    observed = torch.as_tensor(view.observed_y[labeled_indices], dtype=torch.long, device=device)\n    weight = torch.as_tensor(peer_probability[labeled_indices], dtype=torch.float32, device=device)\n    with torch.no_grad():\n        px = (model(x1).softmax(1)+model(x2).softmax(1))/2\n        tx = refine_targets(observed, px, weight, config["temperature"])\n        pu = (model(u1).softmax(1)+model(u2).softmax(1)+peer(u1).softmax(1)+peer(u2).softmax(1))/4\n        tu = sharpen(pu, config["temperature"]).detach()\n    inputs = torch.cat([x1, x2, u1, u2]); target = torch.cat([tx, tx, tu, tu])\n    inputs, target = mixup(inputs, target, config["mixup_alpha"], rng, generator)\n    logits = model(inputs)\n    n = len(labeled_indices)*2\n    lx, lu, reg = semisupervised_loss(logits[:n], target[:n], logits[n:], target[n:])\n    loss = lx+config["lambda_u"]*ramp_fraction*lu+config["regularizer_weight"]*reg\n    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()\n    return {"loss": float(loss.detach()), "labeled_ce": float(lx.detach()), "unlabeled_mse": float(lu.detach()),\n            "prior_regularization": float(reg.detach()), "labeled_examples": len(labeled_indices), "unlabeled_examples": len(unlabeled_indices)}\n', encoding='utf-8')


In [ ]:
from pathlib import Path
destination = Path('/kaggle/working/robust_learning_research') / 'src/robust_learning/strong/vendor_dividemix/PreResNet.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text("import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom torch.autograd import Variable\n\n\ndef conv3x3(in_planes, out_planes, stride=1):\n    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)\n\n\nclass BasicBlock(nn.Module):\n    expansion = 1\n\n    def __init__(self, in_planes, planes, stride=1):\n        super(BasicBlock, self).__init__()\n        self.conv1 = conv3x3(in_planes, planes, stride)\n        self.bn1 = nn.BatchNorm2d(planes)\n        self.conv2 = conv3x3(planes, planes)\n        self.bn2 = nn.BatchNorm2d(planes)\n\n        self.shortcut = nn.Sequential()\n        if stride != 1 or in_planes != self.expansion*planes:\n            self.shortcut = nn.Sequential(\n                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),\n                nn.BatchNorm2d(self.expansion*planes)\n            )\n\n    def forward(self, x):\n        out = F.relu(self.bn1(self.conv1(x)))\n        out = self.bn2(self.conv2(out))\n        out += self.shortcut(x)\n        out = F.relu(out)\n        return out\n\n\nclass PreActBlock(nn.Module):\n    '''Pre-activation version of the BasicBlock.'''\n    expansion = 1\n\n    def __init__(self, in_planes, planes, stride=1):\n        super(PreActBlock, self).__init__()\n        self.bn1 = nn.BatchNorm2d(in_planes)\n        self.conv1 = conv3x3(in_planes, planes, stride)\n        self.bn2 = nn.BatchNorm2d(planes)\n        self.conv2 = conv3x3(planes, planes)\n\n        self.shortcut = nn.Sequential()\n        if stride != 1 or in_planes != self.expansion*planes:\n            self.shortcut = nn.Sequential(\n                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False)\n            )\n\n    def forward(self, x):\n        out = F.relu(self.bn1(x))\n        shortcut = self.shortcut(out)\n        out = self.conv1(out)\n        out = self.conv2(F.relu(self.bn2(out)))\n        out += shortcut\n        return out\n\n\nclass Bottleneck(nn.Module):\n    expansion = 4\n\n    def __init__(self, in_planes, planes, stride=1):\n        super(Bottleneck, self).__init__()\n        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)\n        self.bn1 = nn.BatchNorm2d(planes)\n        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)\n        self.bn2 = nn.BatchNorm2d(planes)\n        self.conv3 = nn.Conv2d(planes, self.expansion*planes, kernel_size=1, bias=False)\n        self.bn3 = nn.BatchNorm2d(self.expansion*planes)\n\n        self.shortcut = nn.Sequential()\n        if stride != 1 or in_planes != self.expansion*planes:\n            self.shortcut = nn.Sequential(\n                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),\n                nn.BatchNorm2d(self.expansion*planes)\n            )\n\n    def forward(self, x):\n        out = F.relu(self.bn1(self.conv1(x)))\n        out = F.relu(self.bn2(self.conv2(out)))\n        out = self.bn3(self.conv3(out))\n        out += self.shortcut(x)\n        out = F.relu(out)\n        return out\n\n\nclass PreActBottleneck(nn.Module):\n    '''Pre-activation version of the original Bottleneck module.'''\n    expansion = 4\n\n    def __init__(self, in_planes, planes, stride=1):\n        super(PreActBottleneck, self).__init__()\n        self.bn1 = nn.BatchNorm2d(in_planes)\n        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)\n        self.bn2 = nn.BatchNorm2d(planes)\n        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)\n        self.bn3 = nn.BatchNorm2d(planes)\n        self.conv3 = nn.Conv2d(planes, self.expansion*planes, kernel_size=1, bias=False)\n\n        self.shortcut = nn.Sequential()\n        if stride != 1 or in_planes != self.expansion*planes:\n            self.shortcut = nn.Sequential(\n                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False)\n            )\n\n    def forward(self, x):\n        out = F.relu(self.bn1(x))\n        shortcut = self.shortcut(out)\n        out = self.conv1(out)\n        out = self.conv2(F.relu(self.bn2(out)))\n        out = self.conv3(F.relu(self.bn3(out)))\n        out += shortcut\n        return out\n\n\nclass ResNet(nn.Module):\n    def __init__(self, block, num_blocks, num_classes=10):\n        super(ResNet, self).__init__()\n        self.in_planes = 64\n\n        self.conv1 = conv3x3(3,64)\n        self.bn1 = nn.BatchNorm2d(64)\n        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)\n        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)\n        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)\n        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)\n        self.linear = nn.Linear(512*block.expansion, num_classes)\n\n    def _make_layer(self, block, planes, num_blocks, stride):\n        strides = [stride] + [1]*(num_blocks-1)\n        layers = []\n        for stride in strides:\n            layers.append(block(self.in_planes, planes, stride))\n            self.in_planes = planes * block.expansion\n        return nn.Sequential(*layers)\n\n    def forward(self, x, lin=0, lout=5):\n        out = x\n        if lin < 1 and lout > -1:\n            out = self.conv1(out)\n            out = self.bn1(out)\n            out = F.relu(out)\n        if lin < 2 and lout > 0:\n            out = self.layer1(out)\n        if lin < 3 and lout > 1:\n            out = self.layer2(out)\n        if lin < 4 and lout > 2:\n            out = self.layer3(out)\n        if lin < 5 and lout > 3:\n            out = self.layer4(out)\n        if lout > 4:\n            out = F.avg_pool2d(out, 4)\n            out = out.view(out.size(0), -1)\n            out = self.linear(out)\n        return out\n\n\ndef ResNet18(num_classes=10):\n    return ResNet(PreActBlock, [2,2,2,2], num_classes=num_classes)\n\ndef ResNet34(num_classes=10):\n    return ResNet(BasicBlock, [3,4,6,3], num_classes=num_classes)\n\ndef ResNet50(num_classes=10):\n    return ResNet(Bottleneck, [3,4,6,3], num_classes=num_classes)\n\ndef ResNet101(num_classes=10):\n    return ResNet(Bottleneck, [3,4,23,3], num_classes=num_classes)\n\ndef ResNet152(num_classes=10):\n    return ResNet(Bottleneck, [3,8,36,3], num_classes=num_classes)\n\n\ndef test():\n    net = ResNet18()\n    y = net(Variable(torch.randn(1,3,32,32)))\n    print(y.size())\n", encoding='utf-8')


In [ ]:
from pathlib import Path
destination = Path('/kaggle/working/robust_learning_research') / 'src/robust_learning/strong/vendor_dividemix/LICENSE'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text('MIT License\n\nCopyright (c) 2020 Junnan Li\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', encoding='utf-8')


In [ ]:
from pathlib import Path
destination = Path('/kaggle/working/robust_learning_research') / 'scripts/run_kaggle_gpu_pilot.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text('"""Bounded, timing-only CIFAR-10 CUDA pilot for DSS+ and DivideMix.\n\nThis runner does not evaluate accuracy, read the test set, or launch a campaign.\nThe Kaggle notebook embeds this script and the independently implemented modules.\n"""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torchvision.datasets import CIFAR10\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / "src"))\nfrom robust_learning.strong.cifar import ImageTrainingView, batch_images, corrupt, image_transforms, make_splits\nfrom robust_learning.strong.equations import fit_clean_probability, peer_policies\nfrom robust_learning.strong.kernels import DSSState, dividemix_update, dss_update\nfrom robust_learning.strong.vendor_dividemix.PreResNet import ResNet18\n\n\nclass Censored(RuntimeError):\n    pass\n\n\ndef batches(indices, batch_size):\n    """Visit each ID exactly once, including the final short batch."""\n    for start in range(0, len(indices), batch_size):\n        yield indices[start:start + batch_size]\n\n\ndef guard(deadline):\n    if time.perf_counter() >= deadline:\n        raise Censored("shared 30-minute pilot deadline")\n\n\ndef measured(name, operation):\n    torch.cuda.synchronize()\n    torch.cuda.reset_peak_memory_stats()\n    start = time.perf_counter()\n    try:\n        result = operation()\n        torch.cuda.synchronize()\n        return {"stage": name, "seconds": time.perf_counter() - start,\n                "peak_allocated_mib": torch.cuda.max_memory_allocated() / 2**20,\n                "peak_reserved_mib": torch.cuda.max_memory_reserved() / 2**20,\n                **result}\n    except Exception as error:\n        error.stage_measurement = {"stage": name, "status": "incomplete",\n                                   "seconds_before_failure": time.perf_counter() - start,\n                                   "peak_allocated_mib": torch.cuda.max_memory_allocated() / 2**20,\n                                   "peak_reserved_mib": torch.cuda.max_memory_reserved() / 2**20}\n        raise\n\n\ndef load_training(cifar_root, config):\n    # CIFAR10 checks the official batch MD5s. An attached, extracted dataset is\n    # required; no download, network access, or dataset upload occurs here.\n    dataset = CIFAR10(root=str(cifar_root), train=True, download=False)\n    labels = np.asarray(dataset.targets, dtype=np.int64)\n    split = make_splits(labels, config["split_seed"], config["validation_per_class"], config["imbalance_factor"])\n    ids = split["train"]\n    observed, _, _ = corrupt(labels[ids], config["noise_seed"], config["noise_rate"], config["noise_kind"])\n    prior_counts = np.bincount(labels[split["prior"]], minlength=10)\n    prior = torch.as_tensor(prior_counts / prior_counts.sum(), dtype=torch.float32)\n    view = ImageTrainingView(dataset.data[ids].copy(), observed.copy(), ids.copy())\n    provenance = {"train_count": len(ids), "prior_calibration_count": len(split["prior"]),\n                  "observed_labels_sha256": hashlib.sha256(observed.tobytes()).hexdigest(),\n                  "train_ids_sha256": hashlib.sha256(ids.tobytes()).hexdigest(),\n                  "clean_calibration_prior": prior.tolist(),\n                  "model_validation_opened": False, "test_opened": False,\n                  "training_oracle_exposed_to_models": False}\n    del dataset, labels, observed, ids, split\n    return view, prior, provenance\n\n\ndef model_pair(seed, settings, device):\n    models, optimizers = [], []\n    for net in range(2):\n        torch.manual_seed(seed + 10000 * net)\n        model = ResNet18(num_classes=10).to(device)\n        models.append(model)\n        optimizers.append(torch.optim.SGD(model.parameters(), lr=settings["lr"], momentum=settings["momentum"],\n                                          weight_decay=settings["weight_decay"]))\n    return models, optimizers\n\n\ndef epoch_order(count, seed, phase):\n    return np.random.default_rng(np.random.SeedSequence([seed, phase, 913])).permutation(count)\n\n\ndef warmup_dss(models, optimizers, states, view, weak, strong, settings, batch_size, seed, deadline, device, progress):\n    order = epoch_order(len(view.ids), seed, 0)\n    observed = torch.as_tensor(view.observed_y, dtype=torch.long)\n    rng = np.random.default_rng(seed + 8000)\n    generator = torch.Generator().manual_seed(seed + 9000)\n    updates = 0\n    for net in range(2):\n        for ix in batches(order, batch_size):\n            guard(deadline)\n            x = batch_images(view, ix, weak).to(device)\n            s = batch_images(view, ix, strong).to(device)\n            dss_update(models[net], optimizers[net], states[net], ix, x, observed[ix].to(device),\n                       torch.ones(len(ix), dtype=torch.bool), torch.zeros(len(ix), 10, dtype=torch.bool),\n                       s, settings, rng, generator)\n            updates += 1\n            progress.update(network=net, network_updates=updates, images_processed=progress.get("images_processed", 0) + len(ix))\n        states[net].finish_epoch()\n    return {"images_per_network": len(order), "network_updates": updates, "completed_full_epoch": True,\n            "host_history_mib": sum(h.numel() * h.element_size() for state in states for h in state.history) / 2**20}\n\n\ndef mature_dss(models, optimizers, states, view, weak, strong, settings, batch_size, seed, deadline, device, progress):\n    order = epoch_order(len(view.ids), seed, 1)\n    observed = torch.as_tensor(view.observed_y, dtype=torch.long)\n    # Freeze both policies before either learner updates; each receives its peer\'s.\n    policies = peer_policies(*(state.policy(observed) for state in states))\n    rng = np.random.default_rng(seed + 8100)\n    generator = torch.Generator().manual_seed(seed + 9100)\n    updates, accepted = 0, []\n    for net in range(2):\n        keep, exclude = policies[net]\n        accepted.append(int(keep.sum()))\n        for ix in batches(order, batch_size):\n            guard(deadline)\n            x = batch_images(view, ix, weak).to(device)\n            s = batch_images(view, ix, strong).to(device)\n            dss_update(models[net], optimizers[net], states[net], ix, x, observed[ix].to(device),\n                       keep[ix], exclude[ix], s, settings, rng, generator)\n            updates += 1\n            progress.update(network=net, network_updates=updates, images_processed=progress.get("images_processed", 0) + len(ix))\n        states[net].finish_epoch()\n    return {"images_per_network": len(order), "network_updates": updates,\n            "accepted_labels_per_peer_policy": accepted, "completed_full_epoch": True,\n            "ccs_history_epochs_before_mature_operation": 1,\n            "host_history_mib": sum(h.numel() * h.element_size() for state in states for h in state.history) / 2**20}\n\n\ndef warmup_dividemix(models, optimizers, view, weak, batch_size, seed, deadline, device, progress):\n    order = epoch_order(len(view.ids), seed, 0)\n    observed = torch.as_tensor(view.observed_y, dtype=torch.long)\n    updates = 0\n    for net in range(2):\n        model, optimizer = models[net], optimizers[net]\n        for ix in batches(order, batch_size):\n            guard(deadline)\n            x = batch_images(view, ix, weak).to(device)\n            model.train()\n            optimizer.zero_grad(set_to_none=True)\n            loss = F.cross_entropy(model(x), observed[ix].to(device))\n            loss.backward()\n            optimizer.step()\n            updates += 1\n            progress.update(network=net, network_updates=updates, images_processed=progress.get("images_processed", 0) + len(ix))\n    return {"images_per_network": len(order), "network_updates": updates, "completed_full_epoch": True}\n\n\ndef scan_dividemix(models, view, inference, batch_size, seed, deadline, device, progress):\n    observed = torch.as_tensor(view.observed_y, dtype=torch.long)\n    probabilities, fits = [], []\n    for net, model in enumerate(models):\n        model.eval()\n        losses = np.empty(len(observed), dtype=np.float64)\n        with torch.no_grad():\n            for ix in batches(np.arange(len(observed)), batch_size):\n                guard(deadline)\n                x = batch_images(view, ix, inference).to(device)\n                losses[ix] = F.cross_entropy(model(x), observed[ix].to(device), reduction="none").cpu().numpy()\n                progress.update(network=net, images_scanned=progress.get("images_scanned", 0) + len(ix))\n        start = time.perf_counter()\n        probabilities.append(fit_clean_probability(losses, seed + net))\n        fits.append({"network": net, "gmm_seconds": time.perf_counter() - start})\n    return probabilities, {"images_per_network": len(observed), "fits": fits,\n                           "completed_full_scan": True}\n\n\ndef mature_dividemix(models, optimizers, view, probabilities, weak, settings, batch_size, seed, deadline, progress):\n    rng = np.random.default_rng(seed + 8200)\n    generator = torch.Generator().manual_seed(seed + 9200)\n    partitions = []\n    for net in range(2):\n        peer_prob = probabilities[1 - net]\n        labeled = np.flatnonzero(peer_prob > settings["clean_threshold"])\n        unlabeled = np.flatnonzero(peer_prob <= settings["clean_threshold"])\n        if not len(labeled) or not len(unlabeled):\n            raise Censored("empty GMM partition; no synthetic fallback")\n        partitions.append((labeled, unlabeled, peer_prob))\n    counts = []\n    for net, (labeled, unlabeled, peer_prob) in enumerate(partitions):\n        labeled_order = rng.permutation(labeled)\n        unlabeled_order = rng.permutation(unlabeled)\n        cursor, updates, used_unlabeled = 0, 0, 0\n        for ix in batches(labeled_order, batch_size):\n            guard(deadline)\n            # Cycle only the unlabeled pool, as the official MixMatch loader does.\n            if cursor + len(ix) > len(unlabeled_order):\n                unlabeled_order = rng.permutation(unlabeled)\n                cursor = 0\n            iu = unlabeled_order[cursor:cursor + len(ix)]\n            if len(iu) < len(ix):\n                iu = rng.choice(unlabeled, len(ix), replace=True)\n            cursor += len(ix)\n            dividemix_update(models[net], models[1 - net], optimizers[net], view, ix, iu,\n                             peer_prob, weak, settings, rng, generator, ramp_fraction=1.0)\n            updates += 1\n            used_unlabeled += len(iu)\n            progress.update(network=net, network_updates=progress.get("network_updates", 0) + 1,\n                            labeled_seen=progress.get("labeled_seen", 0) + len(ix),\n                            unlabeled_draws=progress.get("unlabeled_draws", 0) + len(iu))\n        counts.append({"network": net, "labeled_partition": len(labeled), "unlabeled_partition": len(unlabeled),\n                       "labeled_seen_once": len(labeled_order), "unlabeled_draws": used_unlabeled,\n                       "network_updates": updates})\n    return {"partitions": counts, "completed_full_partition_epoch": True,\n            "mature_loss_ramp_fraction": 1.0}\n\n\ndef run(args):\n    if not 0 < args.time_limit_seconds <= 1800 or args.batch_size < 1:\n        raise ValueError("time limit must be <=1800 seconds and batch size positive")\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU required; no CPU fallback")\n    output = args.output.resolve()\n    output.mkdir(parents=True, exist_ok=False)\n    start = time.perf_counter()\n    deadline = start + args.time_limit_seconds\n    config = json.loads((ROOT / "configs" / "strong_baseline_pilot.json").read_text())\n    device = torch.device("cuda:0")\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.manual_seed(config["pilot_seed"])\n    torch.cuda.manual_seed_all(config["pilot_seed"])\n    view, prior, provenance = load_training(args.cifar_root, config)\n    weak, strong, inference = image_transforms()\n    report = {"purpose": "timing_and_memory_only", "quality_evaluation_executed": False,\n              "full_training_campaign_executed": False, "pilot_limit_seconds": args.time_limit_seconds,\n              "batch_size": args.batch_size, "seed": config["pilot_seed"], "data": provenance,\n              "environment": {"python": platform.python_version(), "torch": torch.__version__,\n                              "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),\n                              "gpu_total_mib": torch.cuda.get_device_properties(0).total_memory / 2**20},\n              "conventions": {"dss_plus_target": "paper_raw_weak_p_to_power_1_over_T",\n                              "dividemix_mse": "official_code_class_mean_lambda_25",\n                              "dividemix_profile_ramp": "one_for_operation_cost_only"},\n              "methods": {}, "elapsed_seconds": None}\n    report_path = output / "gpu_pilot_report.json"\n    def save():\n        report["elapsed_seconds"] = time.perf_counter() - start\n        report_path.write_text(json.dumps(report, indent=2, allow_nan=False))\n    save()\n    for method in ("dss_plus_estimated_prior", "dividemix"):\n        record = {"status": "started", "stages": [], "progress": {}}\n        report["methods"][method] = record\n        save()\n        models = optimizers = states = probabilities = None\n        try:\n            guard(deadline)\n            settings = config["dss_plus" if method.startswith("dss") else "dividemix"]\n            models, optimizers = model_pair(config["pilot_seed"], settings, device)\n            if method.startswith("dss"):\n                states = [DSSState(len(view.ids), 10, prior) for _ in range(2)]\n                operations = [("warmup_full_epoch", lambda: warmup_dss(models, optimizers, states, view, weak, strong, settings,\n                                                                        args.batch_size, config["pilot_seed"], deadline, device, record["progress"])),\n                              ("mature_operation_full_epoch", lambda: mature_dss(models, optimizers, states, view, weak, strong, settings,\n                                                                                   args.batch_size, config["pilot_seed"], deadline, device, record["progress"]))]\n            else:\n                operations = [("warmup_full_epoch", lambda: warmup_dividemix(models, optimizers, view, weak,\n                                                                              args.batch_size, config["pilot_seed"], deadline, device, record["progress"]))]\n            for name, operation in operations:\n                guard(deadline)\n                record["active_stage"] = name\n                record["progress"] = {}\n                save()\n                record["stages"].append(measured(name, operation))\n                record.pop("active_stage", None)\n                save()\n            if method == "dividemix":\n                guard(deadline)\n                record["active_stage"] = "full_data_gmm_scan"\n                record["progress"] = {}\n                save()\n                holder = {}\n                def scan():\n                    holder["probabilities"], details = scan_dividemix(models, view, inference, args.batch_size,\n                                                                       config["pilot_seed"], deadline, device, record["progress"])\n                    return details\n                record["stages"].append(measured("full_data_gmm_scan", scan))\n                probabilities = holder["probabilities"]\n                record.pop("active_stage", None)\n                save()\n                record["active_stage"] = "mature_operation_full_epoch"\n                record["progress"] = {}\n                save()\n                record["stages"].append(measured("mature_operation_full_epoch", lambda: mature_dividemix(\n                    models, optimizers, view, probabilities, weak, settings, args.batch_size, config["pilot_seed"], deadline, record["progress"])))\n                record.pop("active_stage", None)\n                save()\n            record["status"] = "completed_timing_pilot"\n        except (Censored, torch.cuda.OutOfMemoryError) as error:\n            if hasattr(error, "stage_measurement"):\n                record["stages"].append(error.stage_measurement)\n            record.update(status="censored", reason=f"{type(error).__name__}: {error}")\n        except Exception as error:\n            if hasattr(error, "stage_measurement"):\n                record["stages"].append(error.stage_measurement)\n            record.update(status="failed", reason=f"{type(error).__name__}: {error}")\n        finally:\n            record.pop("active_stage", None)\n            save()\n            del models, optimizers, states, probabilities\n            torch.cuda.empty_cache()\n    report["budget_exceeded_at_boundary"] = report["elapsed_seconds"] > args.time_limit_seconds\n    save()\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--cifar-root", type=Path, required=True,\n                        help="attached Kaggle dataset directory containing cifar-10-batches-py")\n    parser.add_argument("--output", type=Path, required=True, help="fresh /kaggle/working output directory")\n    parser.add_argument("--batch-size", type=int, default=128)\n    parser.add_argument("--time-limit-seconds", type=int, default=1800)\n    run(parser.parse_args())\n', encoding='utf-8')


In [ ]:
import hashlib
import json
from pathlib import Path

project = Path('/kaggle/working/robust_learning_research')
manifest = [{'path': 'configs/strong_baseline_pilot.json', 'sha256': '5c65a795646824b2253e1652d61dccba428f103713c9f1060c5b0bf919f56b9b'}, {'path': 'src/robust_learning/__init__.py', 'sha256': 'e7836c625f68d6429b20986000dc54cccef8a06f00f32e51f8e21fe3ac93a68d'}, {'path': 'src/robust_learning/strong/__init__.py', 'sha256': 'b22acce4a213d6a39956caeeeb7c8667fc32d1c04b813eb13b5882819581f2ef'}, {'path': 'src/robust_learning/strong/cifar.py', 'sha256': 'dc5c845b3d8e666fb18fe1d14aa3600a5de4eef9a8dabb31ea9461955e1baad9'}, {'path': 'src/robust_learning/strong/equations.py', 'sha256': '53beaff626ae113502940003a6135942718a8ec0b409553b2d6a82e69ba47a68'}, {'path': 'src/robust_learning/strong/kernels.py', 'sha256': '71b4627355c380f18a3c9b663eac7309d78a951706b5bfc28262ed4d61a4a305'}, {'path': 'src/robust_learning/strong/vendor_dividemix/PreResNet.py', 'sha256': '7d9906f22f821d0d26be0850ab7ebfda22dfabe29d78e02a4b7077494c0ec26b'}, {'path': 'src/robust_learning/strong/vendor_dividemix/LICENSE', 'sha256': 'cc0ac28150c58ebbc876c304713435a92666ab8c3bf70f9dc38f8c625314a23e'}, {'path': 'scripts/run_kaggle_gpu_pilot.py', 'sha256': '89c3553a30cb374cb10e8bb39df7170b41820604dbd59ef32f8893995b89431e'}]
for item in manifest:
    actual = hashlib.sha256((project / item['path']).read_bytes()).hexdigest()
    assert actual == item['sha256'], item['path']
print('Embedded source hashes verified:', len(manifest))


In [ ]:
from pathlib import Path
import sys

# Edit this if more than one attached dataset contains the official CIFAR-10 batches.
candidates = sorted(p.parent for p in Path('/kaggle/input').rglob('cifar-10-batches-py') if p.is_dir())
if len(candidates) != 1:
    raise RuntimeError(f'Expected exactly one attached extracted CIFAR-10 dataset; found {candidates}')
cifar_root = candidates[0]
sys.path.insert(0, '/kaggle/working/robust_learning_research/scripts')
from run_kaggle_gpu_pilot import run
from argparse import Namespace

report = run(Namespace(cifar_root=cifar_root,
                       output=Path('/kaggle/working/robust-learning-gpu-pilot'),
                       batch_size=128, time_limit_seconds=1800))
print('Report:', '/kaggle/working/robust-learning-gpu-pilot/gpu_pilot_report.json')
print({name: entry['status'] for name, entry in report['methods'].items()})
